# Odd-Even Transit Depth Check

**Purpose**: Perform odd-even transit depth validation on each target in Person1's candidate list (`master_candidates.csv`) to identify eclipsing binary (EB) false positives.

**Physical Principle**: Eclipsing binaries often consist of two stars of different brightness, so the primary eclipse (brighter star occulted) and secondary eclipse (fainter star occulted) have different depths. When folded at the orbital period, alternating transits correspond to primary and secondary eclipses appearing in turn. A genuine planet has the same depth every transit, whereas an EB shows a systematic difference between odd- and even-numbered transits.

**Workflow**:
```
1. Download light curve      lightkurve fetches TESS data by Target + Sector (multi-sector stitching supported)
2. Preprocessing              remove_nans -> normalize -> flatten (flatten baseline to ~1.0)
3. Locate transit epoch t0    CSV lacks t0: run narrow-grid BLS around the given Period to refine t0/duration
4. Split odd/even & measure   astropy compute_stats directly returns depth_odd / depth_even with uncertainties
5. Compute significance sigma sigma = |d_odd - d_even| / sqrt(e_odd^2 + e_even^2)
6. Decision                   sigma > 3 -> fail (suspected binary); otherwise pass
```

**Output**: Four new columns added to the original table: `odd_even_sigma`, `n_transits`, `odd_even_reason`, `odd_even_pass`. Saved as `odd_even_result.csv` (Person1's original columns are never modified).

## 0. Install Dependencies

In [ ]:
%pip install -q numpy pandas astropy lightkurve

## 1. Configuration and Data Loading

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
from collections import Counter

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# ---------------- Parameters ----------------
ODD_EVEN_SIGMA_THRESHOLD = 3.0     # sigma > 3 -> fail (suspected binary)
STITCH_ALL_SECTORS = True          # stitch all available sectors -> more transits, more robust check
TESS_CACHE_DIR = 'tess_cache'      # light curve download cache directory
os.makedirs(TESS_CACHE_DIR, exist_ok=True)

# ---------------- Input/Output ----------------
_PATHS = ['master_candidates.csv', 'attachments/master_candidates.csv']
INPUT_CSV = next((p for p in _PATHS if os.path.exists(p)), _PATHS[0])
OUTPUT_CSV = 'odd_even_result.csv'

ORIG_DF = pd.read_csv(INPUT_CSV)   # original data: read-only
original_columns = list(ORIG_DF.columns)
df = ORIG_DF.copy()                # working copy: only append new columns
print(f'Loaded {INPUT_CSV} -> {len(df)} candidates')
df.head()

## 2. Core Functions

`compute_odd_even_sigma` returns `sigma`, a human-readable failure `reason`, and the number of effective transits `n_transits` for each target, making it easy to diagnose *why* a given candidate is "undetermined".

In [ ]:
import lightkurve as lk
from astropy.timeseries import BoxLeastSquares

_LC_CACHE = {}

def get_lightcurve(target, sector):
    '''Return (lightcurve, note). note in {ok, no_search_result, download_failed}.'''
    key = (str(target), int(sector))
    if key in _LC_CACHE:
        return _LC_CACHE[key]
    search = None
    # prefer the specified sector from SPOC; degrade progressively to any source
    for kw in ({'sector': int(sector), 'author': 'SPOC'},
               {'author': 'SPOC'},
               {'sector': int(sector)},
               {}):
        try:
            sr = lk.search_lightcurve(str(target), **kw)
            if sr is not None and len(sr) > 0:
                search = sr
                break
        except Exception:
            continue
    if search is None:
        _LC_CACHE[key] = (None, 'no_search_result')
        return _LC_CACHE[key]
    lc = None
    try:
        if STITCH_ALL_SECTORS and len(search) > 1:
            lc = search.download_all(download_dir=TESS_CACHE_DIR).stitch()   # multi-sector stitching
        else:
            lc = search[0].download(download_dir=TESS_CACHE_DIR)
    except Exception:
        try:
            lc = search[0].download(download_dir=TESS_CACHE_DIR)
        except Exception:
            lc = None
    note = 'ok' if lc is not None else 'download_failed'
    _LC_CACHE[key] = (lc, note)
    return _LC_CACHE[key]

def compute_odd_even_sigma(target, sector, period, duration_hr):
    '''Return dict: {sigma, reason, n_transits, depth_odd, depth_even}. sigma valid when reason=='ok'.'''
    out = {'sigma': np.nan, 'reason': '', 'n_transits': 0,
           'depth_odd': np.nan, 'depth_even': np.nan}
    lc, note = get_lightcurve(target, sector)
    if lc is None:
        out['reason'] = note
        return out
    try:
        # Preprocessing: de-NaN, normalize, flatten baseline (window must be >> transit duration to avoid erasing the transit)
        lc = lc.remove_nans().normalize().flatten(window_length=401)
        t = np.asarray(lc.time.value, dtype=float)
        f = np.asarray(lc.flux.value, dtype=float)
        good = np.isfinite(t) & np.isfinite(f)
        t, f = t[good], f[good]
        if len(t) < 200:
            out['reason'] = f'few_cadences({len(t)})'
            return out
        # noise estimate
        mad = np.nanmedian(np.abs(f - np.nanmedian(f)))
        scale = 1.4826 * mad if mad > 0 else np.nanstd(f)
        dy = np.full(len(t), scale if scale > 0 else 1e-3)
        # refine period in a narrow grid to locate t0 and duration
        bls = BoxLeastSquares(t, f, dy=dy)
        p0 = float(period)
        periods = np.linspace(p0 * 0.999, p0 * 1.001, 300)
        dur0 = max(float(duration_hr), 0.5) / 24.0
        durations = np.unique(np.clip(np.array([dur0 * 0.6, dur0, dur0 * 1.4]),
                                      0.5 / 24.0, 8.0 / 24.0))
        res = bls.power(periods, durations, objective='snr')
        j = int(np.argmax(res.power))
        # Odd/even depths: astropy directly returns depth_odd / depth_even each as (value, uncertainty)
        stats = bls.compute_stats(float(res.period[j]),
                                  float(res.duration[j]),
                                  float(res.transit_time[j]))
        n_tr = int(np.sum(np.atleast_1d(stats['per_transit_count']) > 0))
        out['n_transits'] = n_tr
        d_odd,  e_odd  = float(stats['depth_odd'][0]),  float(stats['depth_odd'][1])
        d_even, e_even = float(stats['depth_even'][0]), float(stats['depth_even'][1])
        out['depth_odd'], out['depth_even'] = d_odd * 1e6, d_even * 1e6
        comb = np.sqrt(e_odd ** 2 + e_even ** 2)
        if not np.isfinite(comb) or comb <= 0:
            out['reason'] = f'oddeven_err_undefined(n_transits={n_tr})'
            return out
        out['sigma'] = abs(d_odd - d_even) / comb
        out['reason'] = 'ok'
        return out
    except Exception as e:
        out['reason'] = f'exception:{type(e).__name__}:{str(e)[:60]}'
        out['n_transits'] = -1
        return out

## 2.1 Single-Target Connectivity Diagnostic (Run This Cell First)

Use a deep, short-period target with many transits **TOI-1516** (P=2.06d, depth=12867ppm, SNR=209) to verify the **download + BLS** pipeline. If this cell fails, the problem is **network/MAST access**, not the data itself.

In [ ]:
print('lightkurve version:', lk.__version__)
_dbg = ('TOI-1516', 57, 2.0559, 2.45)
try:
    _sr = lk.search_lightcurve(_dbg[0], author='SPOC')
    print('search_lightcurve results found:', 0 if _sr is None else len(_sr))
except Exception as e:
    print('search_lightcurve failed with error:', repr(e))
_res = compute_odd_even_sigma(*_dbg)
print('Diagnostic result:', _res)

r = _res['reason']
if r == 'no_search_result':
    print('>> Conclusion: No data found. MAST unreachable / no network / proxy needed. This is why the batch run shows all NaN.')
elif r == 'download_failed':
    print('>> Conclusion: Search succeeded but download failed. Usually a network interruption or cache directory permission issue; retry or change TESS_CACHE_DIR.')
elif r.startswith('few_cadences'):
    print('>> Conclusion: Download succeeded but too few valid cadences (data quality issue).')
elif r.startswith('oddeven'):
    print('>> Conclusion: Download and BLS are working; this target simply lacks enough transits to split odd/even (data limitation).')
elif r.startswith('exception'):
    print('>> Conclusion: Code/dependency exception; troubleshoot using the exception type shown above (commonly a lightkurve version API mismatch).')
elif r == 'ok':
    print('>> Conclusion: Full pipeline working! Proceed to batch run.')

## 3. Batch Odd-Even Validation

Run each target, printing sigma, effective transit count, and reason in real time; a summary of failure reasons is shown at the end.

In [ ]:
sigmas, reasons, ntr_list, dodd_list, deven_list = [], [], [], [], []
for i, row in df.iterrows():
    r = compute_odd_even_sigma(row['Target'], row['Sector'],
                               row['Period (d)'], row['Duration (hr)'])
    sigmas.append(r['sigma']); reasons.append(r['reason'])
    ntr_list.append(r['n_transits'])
    dodd_list.append(r['depth_odd']); deven_list.append(r['depth_even'])
    tag = 'NaN' if not np.isfinite(r['sigma']) else f"{r['sigma']:.2f}"
    print(f"[{i+1:>2}/{len(df)}] {str(row['Target']):<12} sigma={tag:<6} "
          f"n_transits={r['n_transits']:<3} reason={r['reason']}")

df['depth_odd_ppm']  = dodd_list
df['depth_even_ppm'] = deven_list
df['n_transits']     = ntr_list
df['odd_even_sigma'] = sigmas
df['odd_even_reason'] = reasons
df['odd_even_pass'] = pd.array(
    [(x <= ODD_EVEN_SIGMA_THRESHOLD) if np.isfinite(x) else pd.NA for x in sigmas],
    dtype='boolean')

print('\nOdd-even check completed ->  pass:', int((df['odd_even_pass'] == True).sum()),
      '  fail:', int((df['odd_even_pass'] == False).sum()),
      '  undetermined:', int(df['odd_even_pass'].isna().sum()))
print('Undetermined reason distribution:')
for reason, cnt in Counter([rs for rs, s in zip(reasons, sigmas)
                            if not np.isfinite(s)]).most_common():
    print(f'   {cnt:>3}  x  {reason}')

## 4. Save Results

In [ ]:
# Integrity check: original data must remain unchanged
assert df[original_columns].equals(ORIG_DF[original_columns]), 'Original data was unexpectedly modified!'

new_cols = ['depth_odd_ppm', 'depth_even_ppm', 'n_transits',
            'odd_even_sigma', 'odd_even_reason', 'odd_even_pass']
result = df[original_columns + new_cols].copy()
result.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print('Saved ->', OUTPUT_CSV, ' (original data integrity check passed)')

# Show targets flagged as suspected eclipsing binaries (fail)
failed = result[result['odd_even_pass'] == False]
print(f'\nTargets flagged as fail (sigma > {ODD_EVEN_SIGMA_THRESHOLD}, suspected EB): {len(failed)}')
result

## 5. Notes

- `odd_even_pass`: `True` = odd/even depths consistent (pass), `False` = sigma > 3 suspected binary, `<NA>` = undetermined.
- If many `<NA>`, check the **reason distribution** at end of batch: `no_search_result/download_failed` = network issue; `oddeven_err_undefined` = insufficient transits in a single sector (mitigated by multi-sector stitching).
- Conservative principle: undetermined targets are recorded as missing, never treated as pass.
- The entire workflow only appends new columns; Person1's original fields are never modified.